# Getting Started with Neocortex

This notebook walks through the basics of Neocortex — a Graph-based Retrieval-Augmented Generation (GraphRAG) system. You'll learn how to:

1. Index a small document into a knowledge graph
2. Query the graph with natural language
3. Inspect the retrieved entities, relations, and chunks

**Prerequisites**
- Neo4j running locally (start with `docker compose up -d` from the repo root)
- `OPENAI_API_KEY` set in your `.env` file
- `pip install neocortex python-dotenv`

In [ ]:
import os
import time

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready.")

## Sample Data

We'll use a short, entity-rich description of a fictional robotics company. This gives Neocortex plenty of people, products, and relationships to extract.

In [ ]:
SAMPLE_TEXT = """
Acme Robotics was founded in 2019 by Dr. Elena Vasquez and Marcus Chen in Austin, Texas.
The company specializes in autonomous warehouse robots and has grown to 120 employees.
Their flagship product, the AcmeBot X1, uses LiDAR and computer vision to navigate
complex warehouse environments. It can carry up to 50 kg and operates for 12 hours on
a single charge.

Dr. Vasquez serves as CEO and leads the research division. She holds a PhD in robotics
from MIT and previously worked at Boston Dynamics for eight years, where she led the
warehouse automation team. Marcus Chen is the CTO and oversees all engineering. He was
formerly a senior engineer at Tesla's Gigafactory, specializing in robotic assembly lines.

In 2022, Acme Robotics raised a $45 million Series B round led by Sequoia Capital, with
participation from Andreessen Horowitz and Y Combinator. The funding was used to expand
their Austin headquarters and open a new R&D lab in San Francisco.

The company's biggest client is GlobalMart, a retail giant that has deployed 200 AcmeBot
X1 units across its distribution centers in Dallas, Chicago, and Atlanta. GlobalMart
reported a 35% increase in warehouse efficiency after adopting the AcmeBot system.

In early 2024, Acme announced the AcmeBot X2, featuring improved AI navigation, a
75 kg payload capacity, and swarm coordination capabilities that allow multiple robots
to work together seamlessly. The X2 is being piloted by FreshFoods Inc., a grocery
logistics company based in Portland, Oregon.

Dr. Vasquez recently published a paper in the IEEE Robotics journal titled "Scalable
Multi-Agent Navigation in Dense Warehouse Environments" co-authored with Dr. James
Park, the company's Head of AI Research. Dr. Park joined Acme from Google DeepMind
in 2021 and leads a team of 15 machine learning engineers.

Acme Robotics holds 12 patents related to autonomous navigation and has filed 5 more
in 2024 covering swarm intelligence algorithms. The company is headquartered at 500
Innovation Drive, Austin, TX 78701.
""".strip()

print(f"Sample text: {len(SAMPLE_TEXT)} characters, ~{len(SAMPLE_TEXT.split())} words")

## Initialize GraphRAG

We create a `GraphRAG` instance with:
- `working_dir` — where workspace state is stored
- `domain` — guides the LLM on what kind of entities to extract
- `example_queries` — helps the extraction model understand what questions users will ask
- `entity_types` — the categories of entities to look for

In [ ]:
rag = GraphRAG(
    working_dir="./db/examples_quickstart",
    domain="information about a robotics company, its people, products, investors, and clients",
    example_queries=(
        "Who founded Acme Robotics?\n"
        "What products does the company make?\n"
        "Who are the major investors?"
    ),
    entity_types=["person", "company", "product", "location", "organization"],
)
print("GraphRAG initialized.")

## Index the Document

The `insert()` method chunks the text, extracts entities and relationships via an LLM, and stores everything in the knowledge graph. It returns the count of entities, relations, and chunks created.

In [ ]:
token_tracker.reset()
start = time.perf_counter()

num_entities, num_relations, num_chunks = rag.insert(SAMPLE_TEXT)

elapsed = time.perf_counter() - start
print(f"Indexed in {elapsed:.1f}s")
print(f"  Entities:  {num_entities}")
print(f"  Relations: {num_relations}")
print(f"  Chunks:    {num_chunks}")
print()
print(token_tracker.summary())

## Query the Knowledge Graph

Now we can ask natural-language questions. Neocortex retrieves relevant entities, relations, and text chunks from the graph, then generates an answer.

In [ ]:
token_tracker.reset()
start = time.perf_counter()

response = rag.query(
    "Who founded Acme Robotics and what are their backgrounds?",
    params=QueryParam.balanced(),
)

elapsed = time.perf_counter() - start
print(f"Query completed in {elapsed:.1f}s\n")
print("Answer:")
print(response.response)
print()
print(token_tracker.summary())

## Inspect Retrieved Context

Every query response includes the context that was used to generate the answer. This includes scored entities, relations, and text chunks — sorted by relevance.

In [ ]:
# Entities retrieved
print("=" * 60)
print("ENTITIES")
print("=" * 60)
for entity, score in response.context.entities:
    print(f"  [{entity.type}] {entity.name} (score: {score:.4f})")
    print(f"    {entity.description[:120]}..." if len(entity.description) > 120 else f"    {entity.description}")
    print()

# Relations retrieved
print("=" * 60)
print("RELATIONS")
print("=" * 60)
for relation, score in response.context.relations:
    print(f"  {relation.source} --[{relation.rel_type}]--> {relation.target} (score: {score:.4f})")
    print(f"    {relation.description[:120]}")
    print()

# Text chunks retrieved
print("=" * 60)
print("CHUNKS")
print("=" * 60)
for chunk, score in response.context.chunks:
    preview = chunk.content[:200].replace("\n", " ")
    print(f"  Score: {score:.4f}")
    print(f"  {preview}...")
    print()

## Try Another Query

In [ ]:
token_tracker.reset()
start = time.perf_counter()

response2 = rag.query(
    "What is the relationship between Acme Robotics and GlobalMart?",
    params=QueryParam.balanced(),
)

elapsed = time.perf_counter() - start
print(f"Query completed in {elapsed:.1f}s\n")
print("Answer:")
print(response2.response)
print()
print(f"Context: {len(response2.context.entities)} entities, "
      f"{len(response2.context.relations)} relations, "
      f"{len(response2.context.chunks)} chunks")

## Summary

In this notebook you learned how to:
- Initialize a `GraphRAG` instance with domain-specific configuration
- Index text documents into a knowledge graph with `rag.insert()`
- Query the graph with natural language using `rag.query()`
- Inspect the retrieved entities, relations, and chunks

**Next steps:**
- [02_book_rag.ipynb](02_book_rag.ipynb) — Index a full book and run multi-query benchmarks
- [04_query_modes.ipynb](04_query_modes.ipynb) — Explore different query presets and advanced retrieval options